# Negociação de Pares — Cointegração Não Paramétrica (kernel local linear)

Fase de negociação do terceiro método. Por padrão usa os pares, as direções e as bandwidths da
**formação própria** (`pairs_formation_np_propria.ipynb`, teste de cointegração não paramétrica com
valor crítico simulado). Os pares da cointegração linear (`pairs_formation_np.ipynb`) seguem
disponíveis via `ARQUIVO_PARES`.

### Por que a curva não pode ficar congelada na formação

O kernel só tem informação onde o preço já esteve. Nos pares da cointegração linear,
**~40% dos dias de negociação têm $Q_{\text{indep}}$ fora da faixa observada na formação** —
ali a curva da formação simplesmente não está estimada. Por isso, como na versão dinâmica,
a estimação **invade o período de negociação até o dia anterior ao sinal**.

### O ciclo de um par

| Etapa | O que acontece | Amostra da curva |
|---|---|---|
| **Monitoramento** (todo dia $t$, fora do mercado) | spread $S_t = Q_{\text{dep}}(t) - \hat g_{t-1}(Q_{\text{indep}}(t))$, $z_t = (S_t - \mu)/\sigma$ | formação + negociação até $t-1$ |
| **Sinal** | $\lvert z_t\rvert > \delta$ | — |
| **Veto por suporte** (desligado por padrão) | se a massa de kernel em $Q_{\text{indep}}(t)$ vinda de dados com mais de $L$ dias for $<$ `MASSA_MIN`, a curva ali só "conhece" o próprio movimento recente → não abre. `MASSA_MIN = 0` desde 22/10: não melhorava o resultado | — |
| **Veto de β** | $\hat\beta_{t-1}(Q_{\text{indep}}(t)) <$ `BETA_MIN` (0,50) → a perna vendida de $1/\beta$ ficaria alavancada demais | — |
| **Abertura** | registra o ponto de entrada $x_e = Q_{\text{indep}}(t)$, o nível $\hat g_e = \hat g_{t-1}(x_e)$ e o hedge ratio $\beta_e = \hat b_{t-1}(x_e)$ | formação + negociação até $t-1$ |
| **Durante o trade** | $z^{\text{trade}}_\tau = \big(Q_{\text{dep}}(\tau) - \text{ref}_\tau - \mu\big)/\sigma$, com a referência dada por `MODO_SAIDA` (abaixo) | **congelada** na entrada |
| **Fechamento** | $z^{\text{trade}}$ cruza 0, ou fim do período | — |

**Por que o ponto $t$ fica fora da curva do dia $t$:** com $h$ pequeno a curva passa perto de
cada observação usada; se $Q(t)$ entrasse, ela "explicaria" o próprio desvio que se quer operar.

**Por que a referência é congelada na entrada:** se continuasse sendo atualizada durante o
trade, ela perseguiria o preço e o spread "convergiria" porque a curva andou, não porque os
preços voltaram — convergência falsa.

### `MODO_SAIDA` — contra o que se mede a convergência

| Modo | Referência durante o trade | Hedge ratio | Coerência P&L × spread |
|---|---|---|---|
| **`'tangente'`** (padrão) | reta tangente à curva na entrada: $\hat g_e + \beta_e\,(Q_{\text{indep}} - x_e)$ | $\beta_e$ fixo | **exata**: o spread *é* o valor da carteira hedgeada |
| `'curva'` | a curva congelada $\hat g_{\text{entrada}}(Q_{\text{indep}})$ | $\beta_e$ fixo | aproximada — só vale perto de $x_e$ |
| `'curva_rebalanceada'` | a curva congelada | $\hat g'_{\text{entrada}}(Q_{\text{indep}}(\tau))$, rebalanceado todo dia (extensão do slide 19) | aproximada de 1ª ordem, com turnover |

Por que a tangente é o padrão: a primeira rodada com `'curva'` mostrou que o hedge linear e a
referência curva se descolam — o par "convergia" contra a curva enquanto a carteira com $\beta_e$
fixo não realizava o lucro (descasamento médio de −0,014 por trade, igual ao ganho inteiro do
spread). A tangente usa a curva para **decidir** (nível de equilíbrio local e inclinação local na
entrada) e opera exatamente a carteira que mede. Ver `DECISOES.md`, D14.

> ⚠️ Na rodada completa (1991–2015) a tangente **não** mudou o desempenho em relação à
> curva congelada (Sharpe 0,49 × 0,51): o descasamento se compensa no agregado. Ela é o
> padrão pela coerência interna, não por desempenho — ponto aberto para discussão.

**$\mu$ e $\sigma$ vêm da formação** e ficam fixos: no padrão, o erro de previsão um passo à frente
(curva só com o passado) no $h$ escolhido pela CV unilateral — ver `SIGMA_MODO` na formação.

### Retorno — mesma convenção da cointegração linear (Rad et al., 2016)

Perna comprada sempre de \$1, com o $\beta$ do trade vigente:

| position$(t-1)$ | Carteira | Retorno do dia |
|---|---|---|
| $-1$ | long \$1 em $Q_{\text{dep}}$ / short \$$\beta$ em $Q_{\text{indep}}$ | $+(\Delta Q_{\text{dep}} - \beta\,\Delta Q_{\text{indep}})$ |
| $+1$ | long \$1 em $Q_{\text{indep}}$ / short \$$1/\beta$ em $Q_{\text{dep}}$ | $-(\Delta Q_{\text{dep}} - \beta\,\Delta Q_{\text{indep}})/\beta$ |

> No modo `'tangente'` o P&L é exatamente $-\text{pos}\times\text{escala}\times\Delta S_t$, como na
> versão linear. Nos modos `'curva'`, $\Delta S \approx \Delta Q_{\text{dep}} - \hat g'(Q_{\text{indep}})\,\Delta Q_{\text{indep}}$
> só localmente — se $Q_{\text{indep}}$ andar muito durante o trade, a carteira fica sub ou
> sobre-hedgeada em relação à curva.

**Execução:** decisão no fechamento de $t$, execução a partir de $t+1$ (`np.roll`), como nos
outros dois métodos.

## 1. Setup

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from kernel_np import loclin, monitoramento, residuos_unilaterais

warnings.filterwarnings('ignore')

# === Parâmetros da estratégia (iguais aos da cointegração linear) ===
DELTA     = 2.0    # gatilho de entrada: abre posição quando |z| > DELTA
SIGMA_MIN = 0.01   # exclui pares com sigma patológico (ghost tickers)
# Veto de β: o dimensionamento de Rad compra $1 e vende $1/β quando z > +δ. Com β local pequeno
# a perna vendida explode (houve trades com β = 0,08 → $12 vendidos por $1 comprado).
# PADRÃO desde 2026-10-22: 0,50 ≈ percentil 1 do β do OLS nos mesmos pares, isto é, a faixa de
# hedge ratio que a própria linear considera plausível. Testados: 0,05 → Sharpe 0,58;
# 0,25 → 0,72; 0,50 → 0,82 (CV unilateral, 1991–2015). ⚠️ valor escolhido depois de ver o
# resultado; alternativa a discutir: mudar a convenção de dimensionamento das pernas.
BETA_MIN  = 0.50
# PADRÃO desde 2026-11-02: 0,50 ≤ β ≤ 2 — a perna menor vale ao menos metade da maior (regra simétrica);
# com 'maior_perna' o teto não muda o resultado (1991–2015: 1,11 com e sem teto) mas evita pernas quase sem hedge.
BETA_MAX  = 2.0   # teto do β local na entrada (None = sem teto)

# === Dimensionamento das pernas (DECISOES.md § 9) ===
#   'rad'         → Rad et al. (2016) § 4.2.2: perna comprada sempre $1; com z > 0 vende $1/β
#                   (alavancada quando β < 1). Convenção da cointegração linear.
#   'slides'      → slides 17–18: 1 unidade da dependente contra β da independente, nos dois
#                   sentidos ("ou a posição inversa"); alavancada na independente quando β > 1.
#   'maior_perna' → o hedge dos slides (1 : β) com a MAIOR perna normalizada em $1:
#                   retorno ÷ max(1, β). Sem alavancagem em nenhum dos lados.
# PADRÃO desde 2026-11-02 (DECISOES.md § 9.1): hedge dos slides 17–18 (1 : β nos dois sentidos) com a
# maior perna em $1. Rad (perna comprada sempre $1) alavancava a carteira com β < 1 (z > 0) e β > 1 (z < 0).
DIMENSIONAMENTO = 'maior_perna'

# === Filtro de qualidade de dados — mantido IGUAL ao da linear para a comparação ser pareada ===
# ⚠️ Olha o período de negociação inteiro (look-ahead); na distância já foi zerado (= None).
#    Se for zerado na cointegração linear, zerar aqui também.
MAX_RETORNO_DIA = 0.90

# === Veto por suporte ===
# Massa de kernel em Q_indep(t) vinda só de observações com mais de L_SUPORTE pregões de idade.
# Com K(0) = 1, a massa é um "número efetivo de observações" perto do ponto. Abaixo de
# MASSA_MIN, a curva em Q_indep(t) é determinada pelo próprio movimento recente do preço — o
# equilíbrio ali não está identificado. Com h = ∞ (reta) a massa é o tamanho da amostra e o
# veto nunca dispara.
# DESLIGADO por padrão desde 2026-10-22 (MASSA_MIN = 0): com β ≥ 0,50 o veto não acrescenta nada
# — Sharpe 0,82 com mínimo 10, 0,82 com 5, 0,85 sem veto (1991–2015) — e em mercado com
# tendência barra justamente os níveis de preço novos. O código fica para sensibilidade.
L_SUPORTE  = 20    # idade mínima (pregões) dos dados que contam como suporte
MASSA_MIN  = 0.0   # 0 → veto desligado; o valor antigo era 10

# === Referência de convergência durante o trade (ver tabela no topo) ===
#   'tangente'           → reta tangente à curva na entrada, β fixo (padrão)
#   'curva'              → curva congelada na entrada, β fixo
#   'curva_rebalanceada' → curva congelada, β = inclinação da curva no dia (rebalanceia)
MODO_SAIDA = 'tangente'

# === μ e σ do z (o "metro" do spread) ===
#   'formacao'  → fixos, do erro um passo à frente na formação (padrão)
#   'expandido' → re-estimados todo dia: erros um passo à frente da formação + spreads de
#                 monitoramento até t−1 (mesma natureza: curva até t−1 avaliada no preço de t)
#   'movel'     → idem, só os últimos JANELA_MU_SIGMA valores
# Na abertura, μ e σ do dia são congelados junto com a tangente.
MU_SIGMA_MODO   = 'formacao'
JANELA_MU_SIGMA = 126
T_MIN_RESID     = 126    # = T_MIN_UNILATERAL da formação (primeiro dia com erro um passo à frente)

# === Arquivos ===
# PADRÃO desde 2026-10-24: formação PRÓPRIA do NP (pairs_formation_np_propria.ipynb).
# Pares da cointegração linear: 'output/pares/pares_formados_np.csv' (pairs_formation_np.ipynb).
ARQUIVO_PARES = 'output/pares/pares_formados_np_propria.csv'
SUFIXO        = ''     # anexado aos CSVs de saída: rodadas de sensibilidade não sobrescrevem a principal

# === Convenção do portfólio (série DIÁRIA agregada) ===
COMMITTED_CAPITAL = True   # True → denominador = N pares fixo (pares inativos contribuem 0)

# === Retorno MENSAL (Rad et al., §4.5, eq. 14 e 15) ===
N_PARES_NOMINADOS = 20

# === Filtro de período (mês 'YYYY-MM') — igual ao da linear, para a comparação pareada ===
PERIODO_INICIO = None
PERIODO_FIM    = '2015-12'

META_COLS = ['year', 'semester', 'period', 'day_in_semester', 'total_days_sem']

## 2. Carregando os dados

In [ ]:
pares_raw = pd.read_csv(ARQUIVO_PARES)
df        = pd.read_csv('../../data_bases/base_unificada.csv', parse_dates=['date'])
df        = df.set_index('date')

meses_por_linha = pd.PeriodIndex(df.index, freq='M')

pares = pares_raw.copy()
if PERIODO_INICIO is not None:
    pares = pares[pares['negociacao_inicio'] >= PERIODO_INICIO]
if PERIODO_FIM is not None:
    pares = pares[pares['negociacao_fim'] <= PERIODO_FIM]

print(f"Pares disponíveis : {len(pares_raw):,} ({pares_raw['janela'].nunique()} janelas)")
print(f"Origem dos pares  : {pares_raw['origem'].iloc[0]}")
print(f"Período filtrado  : {PERIODO_INICIO or pares_raw['negociacao_inicio'].min()} → "
      f"{PERIODO_FIM or pares_raw['negociacao_fim'].max()}  "
      f"({len(pares):,} pares, {pares['janela'].nunique()} janelas)")
print(f"Pares com h = ∞   : {pares['h_linear'].mean():.1%} (a curva é a reta, re-estimada até t−1)")
print(f"Modo de saída     : {MODO_SAIDA}")

## 3. Funções

In [ ]:
def simular_par_np(precos_form, precos_neg, dep, indep, h, mu, sigma, delta=None, modo=None):
    """
    Simula a negociação de um par pela cointegração não paramétrica.

    precos_form, precos_neg : DataFrames com colunas dep e indep, sem NaN
    h                       : bandwidth da formação (np.inf → reta)
    mu, sigma               : média e desvio do resíduo da formação (ver SIGMA_MODO)
    modo                    : referência de convergência (default: MODO_SAIDA)

    Retorna (DataFrame diário, lista de eventos de sinal).
    """
    if delta is None:
        delta = DELTA
    if modo is None:
        modo = MODO_SAIDA
    if modo not in ('tangente', 'curva', 'curva_rebalanceada'):
        raise ValueError(f"MODO_SAIDA desconhecido: {modo!r}")

    P0  = precos_form.iloc[0]
    y_f = (precos_form[dep]   / P0[dep]).values
    x_f = (precos_form[indep] / P0[indep]).values
    y_n = (precos_neg[dep]    / P0[dep]).values
    x_n = (precos_neg[indep]  / P0[indep]).values
    T, m  = len(x_f), len(x_n)
    x_all = np.concatenate([x_f, x_n])
    y_all = np.concatenate([y_f, y_n])

    # Monitoramento: curva até t−1, avaliada em x_t — todos os dias de uma vez
    g_mon, b_mon, massa = monitoramento(x_f, y_f, x_n, y_n, h, L_SUPORTE)
    # μ e σ de cada dia (ver MU_SIGMA_MODO)
    if MU_SIGMA_MODO == 'formacao':
        mu_t, sig_t = np.full(m, mu), np.full(m, sigma)
    elif MU_SIGMA_MODO in ('expandido', 'movel'):
        u_f = residuos_unilaterais(x_f, y_f, h, 1, T_MIN_RESID)
        pool = np.concatenate([u_f[np.isfinite(u_f)], y_n - g_mon])   # spreads de monitoramento
        nf0 = int(np.isfinite(u_f).sum())
        mu_t, sig_t = np.empty(m), np.empty(m)
        for k in range(m):
            v = pool[:nf0 + k]                                     # tudo até t−1
            if MU_SIGMA_MODO == 'movel':
                v = v[-JANELA_MU_SIGMA:]
            v = v[np.isfinite(v)]
            mu_t[k], sig_t[k] = (v.mean(), v.std()) if len(v) > 10 else (mu, sigma)
    else:
        raise ValueError(f"MU_SIGMA_MODO desconhecido: {MU_SIGMA_MODO!r}")
    z_mon = (y_n - g_mon - mu_t) / sig_t

    positions = np.zeros(m)
    beta_dia  = np.full(m, np.nan)   # β do trade vigente em cada dia
    z_trade   = np.full(m, np.nan)   # z contra a curva congelada
    eventos   = []

    t = 0
    while t < m:
        zt = z_mon[t]
        if not (np.isfinite(zt) and abs(zt) > delta):
            t += 1
            continue

        if massa[t] < MASSA_MIN:
            motivo = 'suporte'
        elif not np.isfinite(b_mon[t]) or b_mon[t] < BETA_MIN or (BETA_MAX is not None and b_mon[t] > BETA_MAX):
            motivo = 'beta'
        else:
            motivo = 'aprovado'
        eventos.append({'dia': precos_neg.index[t], 'z': zt, 'massa_antiga': massa[t],
                        'beta': b_mon[t], 'motivo': motivo})
        if motivo != 'aprovado':
            t += 1
            continue

        # ── Abertura: congela a referência (amostra até t−1) ──────────────────
        pos    = +1 if zt > 0 else -1     # +1: Q_dep caro → vende o spread
        beta_e = b_mon[t]
        if modo == 'tangente':
            # reta tangente à curva em x_e: o spread é o valor da carteira com β_e fixo
            ref    = g_mon[t] + beta_e * (x_n[t:] - x_n[t])
            betas  = np.full(m - t, beta_e)
        else:
            n = T + t
            ref, b_c, _ = loclin(x_n[t:], x_all[:n], y_all[:n], h)
            if modo == 'curva':
                betas = np.full(m - t, beta_e)
            else:   # 'curva_rebalanceada': β = inclinação da curva congelada no dia;
                    # se degenerar (< BETA_MIN ou NaN), mantém o último β válido
                betas = pd.Series(np.where(np.isfinite(b_c) & (b_c >= BETA_MIN), b_c, np.nan))
                betas = betas.ffill().fillna(beta_e).values
        mu_e, sig_e = mu_t[t], sig_t[t]                # congelados na abertura
        z_c = (y_n[t:] - ref - mu_e) / sig_e           # z_c[0] == z_mon[t]

        k = 0
        while t + k < m:
            if k > 0 and ((pos == +1 and z_c[k] <= 0) or (pos == -1 and z_c[k] >= 0)):
                z_trade[t + k] = z_c[k]
                break                                  # fecha no dia t+k (position = 0)
            positions[t + k] = pos
            beta_dia[t + k]  = betas[k]
            z_trade[t + k]   = z_c[k]
            k += 1
        t += k + 1                                     # no dia do fechamento não reabre

    # ── Retorno: execução em t+1, perna comprada de $1 (Rad et al.) ───────────
    pos_lag     = np.roll(positions, 1); pos_lag[0]  = 0
    beta_lag    = np.roll(beta_dia,  1); beta_lag[0] = np.nan
    d_dep       = np.r_[0.0, np.diff(y_n)]
    d_indep     = np.r_[0.0, np.diff(x_n)]

    ativo   = pos_lag != 0
    retorno = np.zeros(m)
    b       = beta_lag[ativo]
    if DIMENSIONAMENTO == 'rad':
        escala = np.where(pos_lag[ativo] > 0, 1.0 / b, 1.0)
    elif DIMENSIONAMENTO == 'slides':
        escala = np.ones_like(b)
    elif DIMENSIONAMENTO == 'maior_perna':
        escala = 1.0 / np.maximum(1.0, b)
    else:
        raise ValueError(f"DIMENSIONAMENTO desconhecido: {DIMENSIONAMENTO!r}")
    retorno[ativo] = -pos_lag[ativo] * escala * (d_dep[ativo] - b * d_indep[ativo])

    diario = pd.DataFrame({
        'spread_mon':     y_n - g_mon,
        'z':              z_mon,
        'z_trade':        z_trade,
        'massa_antiga':   massa,
        'position':       positions,
        'position_lag':   pos_lag,
        'beta':           beta_dia,
        'retorno_diario': retorno,
    }, index=precos_neg.index)
    return diario, eventos


def contar_trades(positions):
    """Retorna (n_abertos, n_fechados_normal, n_forcados)."""
    pos = np.asarray(positions)
    entradas       = np.sum((pos[1:] != 0) & (pos[:-1] == 0))
    if pos[0] != 0:
        entradas  += 1
    saidas_normais = np.sum((pos[:-1] != 0) & (pos[1:] == 0))
    forcado        = 1 if pos[-1] != 0 else 0
    return int(entradas), int(saidas_normais), int(forcado)

## 4. Simulação completa

Filtros de exclusão do par, iguais aos da versão linear: `sigma < SIGMA_MIN`, ticker ausente,
sem dados, `MAX_RETORNO_DIA`. O β não é filtrado aqui — ele só existe na entrada de cada
trade, e o veto de β age sinal a sinal.

In [ ]:
resultados_pares  = []
retornos_por_data = {}
registros_mensais = []
eventos_todos     = []

n_skip_sigma = n_skip_ticker = n_skip_dados = n_skip_outlier = 0
t0 = time.time()

for _, par in pares.iterrows():
    a1, a2 = par['ativo1'], par['ativo2']
    dep, indep = (a1, a2) if par['direcao'] == 'A~B' else (a2, a1)
    mu, sigma  = par['spread_media'], par['spread_desvio']
    h          = float(par['h'])

    if sigma < SIGMA_MIN:
        n_skip_sigma += 1
        continue
    if a1 not in df.columns or a2 not in df.columns:
        n_skip_ticker += 1
        continue

    mask_form = ((meses_por_linha >= pd.Period(par['formacao_inicio'], freq='M')) &
                 (meses_por_linha <= pd.Period(par['formacao_fim'],    freq='M')))
    mask_neg  = ((meses_por_linha >= pd.Period(par['negociacao_inicio'], freq='M')) &
                 (meses_por_linha <= pd.Period(par['negociacao_fim'],    freq='M')))

    precos_form = df.loc[mask_form, [dep, indep]].dropna()
    precos_neg  = df.loc[mask_neg,  [dep, indep]].dropna()
    if len(precos_form) == 0 or len(precos_neg) == 0:
        n_skip_dados += 1
        continue

    if MAX_RETORNO_DIA is not None and precos_neg.pct_change().max().max() > MAX_RETORNO_DIA:
        n_skip_outlier += 1
        continue

    resultado, eventos = simular_par_np(precos_form, precos_neg, dep, indep, h, mu, sigma)
    n_abertos, n_fechados, n_forcados = contar_trades(resultado['position'].values)

    motivos = pd.Series([e['motivo'] for e in eventos], dtype=object)
    resultados_pares.append({
        'janela':         par['janela'],
        'neg_inicio':     par['negociacao_inicio'],
        'neg_fim':        par['negociacao_fim'],
        'ativo1':         a1,
        'ativo2':         a2,
        'direcao':        par['direcao'],
        'rank':           par['rank'],
        'h_rel':          par['h_rel'],
        'h_linear':       par['h_linear'],
        'sigma':          sigma,
        'n_sinais':       len(eventos),
        'n_veto_suporte': int((motivos == 'suporte').sum()),
        'n_veto_beta':    int((motivos == 'beta').sum()),
        'n_trades':       n_abertos,
        'n_fechados':     n_fechados,
        'n_forcados':     n_forcados,
        'beta_medio':     np.nanmean(resultado['beta']) if n_abertos else np.nan,
        'retorno_total':  resultado['retorno_diario'].sum(),
    })
    for e in eventos:
        eventos_todos.append({'janela': par['janela'], 'ativo1': a1, 'ativo2': a2,
                              'h_linear': par['h_linear'], **e})

    # Retorno mensal do par — insumo de REC/RCC (Rad et al., §4.5)
    mes_idx    = resultado.index.to_period('M').astype(str)
    mensal_par = pd.DataFrame({
        'r_mes':    resultado['retorno_diario'].groupby(mes_idx).sum(),
        'negociou': (resultado['position_lag'] != 0).groupby(mes_idx).max(),
    }).rename_axis('mes').reset_index()
    mensal_par['janela'] = par['janela']
    mensal_par['ativo1'] = a1
    mensal_par['ativo2'] = a2
    registros_mensais.append(mensal_par)

    if COMMITTED_CAPITAL:
        ret_dict = resultado['retorno_diario'].to_dict()
        for date in df.index[mask_neg]:
            retornos_por_data.setdefault(date, []).append(ret_dict.get(date, 0.0))
    else:
        for date, row in resultado.iterrows():
            if row['position_lag'] != 0:
                retornos_por_data.setdefault(date, []).append(row['retorno_diario'])

resultados_df = pd.DataFrame(resultados_pares)
eventos_df    = pd.DataFrame(eventos_todos)
mensal_df     = pd.concat(registros_mensais, ignore_index=True)

print(f"Total de pares no arquivo  : {len(pares):,}")
print(f"  Pulados (sigma < {SIGMA_MIN}) : {n_skip_sigma:,}")
print(f"  Pulados (ticker ausente)  : {n_skip_ticker:,}")
print(f"  Pulados (sem dados)       : {n_skip_dados:,}")
print(f"  Pulados (spike positivo)  : {n_skip_outlier:,}")
print(f"  Simulados com sucesso     : {len(resultados_df):,}")
print(f"Tempo                       : {time.time() - t0:.0f} s")

### 4.1 Diagnóstico dos sinais

Quantos sinais o z de monitoramento gerou e quantos foram vetados — separando os pares em que
a CV escolheu a reta ($h=\infty$, veto de suporte impossível) dos pares com curva.

In [ ]:
def resumo_sinais(ev):
    n = len(ev)
    return pd.Series({
        'sinais':        n,
        'aprovados':     (ev['motivo'] == 'aprovado').mean(),
        'veto_suporte':  (ev['motivo'] == 'suporte').mean(),
        'veto_beta':     (ev['motivo'] == 'beta').mean(),
    })

diag = pd.DataFrame({
    'todos':        resumo_sinais(eventos_df),
    'h = ∞ (reta)': resumo_sinais(eventos_df[eventos_df['h_linear']]),
    'h finito':     resumo_sinais(eventos_df[~eventos_df['h_linear']]),
}).T
display(diag.style.format({'sinais': '{:,.0f}', 'aprovados': '{:.1%}',
                           'veto_suporte': '{:.1%}', 'veto_beta': '{:.1%}'}))

aprov = eventos_df[eventos_df['motivo'] == 'aprovado']
print(f"\nβ na entrada (trades aprovados): mediana {aprov['beta'].median():.3f} | "
      f"p10 {aprov['beta'].quantile(.1):.3f} | p90 {aprov['beta'].quantile(.9):.3f}")

## 5. Resultado por par

In [ ]:
print("=== Estatísticas de trades ===")
print(f"Pares que abriram ≥ 1 trade    : {(resultados_df['n_trades'] > 0).sum():,} de {len(resultados_df):,}")
print(f"Total de trades abertos        : {resultados_df['n_trades'].sum():,}")
print(f"Trades fechados normalmente    : {resultados_df['n_fechados'].sum():,}")
print(f"Trades forçados (fim período)  : {resultados_df['n_forcados'].sum():,}")
if resultados_df['n_trades'].sum() > 0:
    print(f"\nTaxa de convergência           : "
          f"{resultados_df['n_fechados'].sum() / resultados_df['n_trades'].sum():.1%}")

print("\n=== Reta × curva ===")
display(resultados_df.groupby('h_linear').agg(
    pares=('retorno_total', 'size'),
    trades_por_par=('n_trades', 'mean'),
    convergencia=('n_fechados', 'sum'),
    trades=('n_trades', 'sum'),
    retorno_medio_par=('retorno_total', 'mean'),
).assign(convergencia=lambda d: d['convergencia'] / d['trades']).drop(columns='trades'))

## 6. Portfólio e curva de equity (série diária)

Mesma agregação da cointegração linear: com `COMMITTED_CAPITAL = True`, todos os pares das
janelas sobrepostas ativas no dia entram no denominador; pares sem posição contribuem 0.

In [ ]:
portfolio = pd.Series(
    {date: np.mean(rets) for date, rets in retornos_por_data.items()}
).sort_index().rename_axis('date')

equity = (1 + portfolio).cumprod()

print(f"Período do portfólio: {portfolio.index[0].date()} → {portfolio.index[-1].date()}")
print(f"Dias de negociação  : {len(portfolio):,}")
print(f"Retorno diário médio: {portfolio.mean():.4%}")
print(f"Retorno acumulado   : {equity.iloc[-1] - 1:.2%}")

## 7. Retorno mensal — REC e RCC (Rad et al., §4.5)

Mesmo cálculo da cointegração linear: por portfólio (janela) e depois média equiponderada dos
portfólios sobrepostos ativos no mês. REC divide pelos pares que negociaram no mês; RCC pelos
$N_P = 20$ nominados.

In [ ]:
todos_pm = mensal_df.groupby(['janela', 'mes']).size().rename('n_pares')
negoc    = mensal_df[mensal_df['negociou']]

soma_r = (negoc.groupby(['janela', 'mes'])['r_mes'].sum()
               .reindex(todos_pm.index, fill_value=0.0))
n_neg  = (negoc.groupby(['janela', 'mes'])['r_mes'].size()
               .reindex(todos_pm.index, fill_value=0))

port_mes = pd.DataFrame({
    'n_pares':      todos_pm,
    'n_negociaram': n_neg,
    'soma_r':       soma_r,
    'REC':          soma_r / n_neg.where(n_neg > 0),
    'RCC':          soma_r / N_PARES_NOMINADOS,
}).reset_index()

rec_mensal = port_mes.groupby('mes')['REC'].mean()
rcc_mensal = port_mes.groupby('mes')['RCC'].mean()


def estatisticas_mensais(serie, nome):
    s, n = serie.dropna(), serie.dropna().size
    media, dp = s.mean(), s.std(ddof=1)
    return {'Medida': nome, 'N meses': n, 'Média': media, 't-stat': media / (dp / np.sqrt(n)),
            'Desvio-padrão': dp, 'Sharpe (mensal)': media / dp,
            'Assimetria': s.skew(), 'Curtose': s.kurtosis() + 3}


tabela_mensal = pd.DataFrame([
    estatisticas_mensais(rec_mensal, 'REC — capital empregado'),
    estatisticas_mensais(rcc_mensal, 'RCC — capital comprometido'),
]).set_index('Medida')

fmt = {'N meses': '{:.0f}', 'Média': '{:.4f}', 't-stat': '{:.4f}', 'Desvio-padrão': '{:.4f}',
       'Sharpe (mensal)': '{:.4f}', 'Assimetria': '{:.4f}', 'Curtose': '{:.4f}'}
display(tabela_mensal.style.format(fmt))

## 8. Métricas de performance (série diária) — bruto × líquido de renda fixa

In [ ]:
rf = pd.read_csv('../../data_bases/rf_diario.csv', parse_dates=['date']).set_index('date')['rf']
rf = rf.reindex(portfolio.index).ffill().bfill()


def metricas_performance(serie):
    equity  = (1 + serie).cumprod()
    maxacum = equity.cummax()
    n_anos  = len(serie) / 252
    return {
        'Retorno acumulado'    : equity.iloc[-1] - 1,
        'Retorno anualizado'   : equity.iloc[-1] ** (1 / n_anos) - 1,
        'Volatilidade anual'   : serie.std() * np.sqrt(252),
        'Sharpe ratio'         : serie.mean() / serie.std() * np.sqrt(252),
        'Drawdown máximo'      : ((equity - maxacum) / maxacum).min(),
        'Dias com retorno > 0' : (serie > 0).mean(),
    }


metricas = pd.DataFrame({'bruto':   metricas_performance(portfolio),
                         'líquido': metricas_performance(portfolio - rf)})
display(metricas.style.format('{:.4f}'))

## 9. Comparação pareada — linear estática × não paramétrica

Lê os resultados gravados por `../cointegration/pairs_trading_coint.ipynb`. **Pré-requisito:**
ele precisa ter rodado sobre o **mesmo** `pares_formados_coint.csv` de onde vieram os pares
daqui e com o mesmo `PERIODO_INICIO`/`PERIODO_FIM` — senão a comparação não é pareada.

O pareamento par a par usa (`janela`, `ativo1`, `ativo2`).

In [ ]:
PATH_LIN_PORT = '../cointegration/output/resultados/negociacao_portfolio_coint.csv'
PATH_LIN_PAR  = '../cointegration/output/resultados/negociacao_pares_coint.csv'

# index_col=0: o notebook linear grava o índice de datas sem nome
lin_port = pd.read_csv(PATH_LIN_PORT, index_col=0, parse_dates=True)['retorno_diario']
lin_par  = pd.read_csv(PATH_LIN_PAR)

datas = portfolio.index.intersection(lin_port.index)
if len(datas) < 0.95 * len(portfolio):
    print(f"⚠ só {len(datas):,} de {len(portfolio):,} dias em comum — os dois notebooks "
          f"não rodaram sobre o mesmo período. Comparação NÃO pareada.")

comp = pd.DataFrame({'linear estática': metricas_performance(lin_port.loc[datas]),
                     'não paramétrica': metricas_performance(portfolio.loc[datas])})
display(comp.style.format('{:.4f}'))

chave = ['janela', 'ativo1', 'ativo2']
par_a_par = resultados_df.merge(lin_par[chave + ['retorno_total', 'n_trades']],
                                on=chave, suffixes=('_np', '_lin'))
dif = par_a_par['retorno_total_np'] - par_a_par['retorno_total_lin']
print(f"\nPares pareados: {len(par_a_par):,} de {len(resultados_df):,}")
print(f"Diferença de retorno por par (NP − linear): média {dif.mean():+.4f} | "
      f"mediana {dif.median():+.4f} | NP melhor em {(dif > 0).mean():.1%}")
print(par_a_par.assign(dif=dif).groupby('h_linear')['dif'].agg(['size', 'mean', 'median'])
      .rename(index={True: 'h = ∞ (reta re-estimada)', False: 'h finito (curva)'}).to_string())

## 10. Explorando um par específico

Mostra o z de monitoramento (curva até $t-1$), o z do trade (curva congelada na entrada) e os
sinais aprovados (○) e vetados (✕). Por padrão, o par com curva ($h$ finito) que mais negociou.

In [ ]:
alvo = (resultados_df[~resultados_df['h_linear']].nlargest(1, 'n_trades').iloc[0]
        if (~resultados_df['h_linear']).any() else resultados_df.nlargest(1, 'n_trades').iloc[0])
par_info = pares[(pares['janela'] == alvo['janela']) & (pares['ativo1'] == alvo['ativo1'])
                 & (pares['ativo2'] == alvo['ativo2'])].iloc[0]

a1, a2 = par_info['ativo1'], par_info['ativo2']
dep, indep = (a1, a2) if par_info['direcao'] == 'A~B' else (a2, a1)
mf = ((meses_por_linha >= pd.Period(par_info['formacao_inicio'], freq='M')) &
      (meses_por_linha <= pd.Period(par_info['formacao_fim'], freq='M')))
mn = ((meses_por_linha >= pd.Period(par_info['negociacao_inicio'], freq='M')) &
      (meses_por_linha <= pd.Period(par_info['negociacao_fim'], freq='M')))
res, ev = simular_par_np(df.loc[mf, [dep, indep]].dropna(), df.loc[mn, [dep, indep]].dropna(),
                         dep, indep, float(par_info['h']),
                         par_info['spread_media'], par_info['spread_desvio'])
ev = pd.DataFrame(ev)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, gridspec_kw={'height_ratios': [2, 2, 1]})
fig.suptitle(f"{dep} ~ g({indep}) — {par_info['janela']}  h/sd={par_info['h_rel']:.2f}",
             fontsize=11, fontweight='bold')

ax = axes[0]
ax.plot(res.index, res['z'], color='gray', lw=1.0, label='z monitoramento (curva até t−1)')
ax.plot(res.index, res['z_trade'], color='purple', lw=1.6, label=f'z do trade ({MODO_SAIDA}, congelada)')
for s in (+DELTA, -DELTA):
    ax.axhline(s, color='red', lw=0.8, ls='--')
ax.axhline(0, color='black', lw=0.8)
if len(ev):
    ok = ev['motivo'] == 'aprovado'
    ax.scatter(ev.loc[ok, 'dia'], ev.loc[ok, 'z'], marker='o', facecolors='none',
               edgecolors='green', s=60, label='sinal aprovado')
    ax.scatter(ev.loc[~ok, 'dia'], ev.loc[~ok, 'z'], marker='x', color='red', s=50,
               label='sinal vetado')
ax.set_ylabel('z')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(res.index, res['massa_antiga'], color='#00695C', lw=1.3)
ax.axhline(MASSA_MIN, color='red', lw=0.9, ls='--', label=f'MASSA_MIN = {MASSA_MIN:g}')
ax.set_yscale('symlog')
ax.set_ylabel(f'massa de kernel\n(> {L_SUPORTE} dias)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(res.index, (1 + res['retorno_diario']).cumprod(), color='#1565C0', lw=1.5)
ax.axhline(1, color='gray', lw=0.8, ls='--')
ax.set_ylabel('equity do par')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%y'))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'output/imagens/par_exemplo_np{SUFIXO}.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Salvando resultados

In [ ]:
S = SUFIXO
resultados_df.to_csv(f'output/resultados/negociacao_pares_np{S}.csv', index=False)
eventos_df.to_csv(f'output/resultados/eventos_sinal_np{S}.csv', index=False)
portfolio.to_frame('retorno_diario').to_csv(f'output/resultados/negociacao_portfolio_np{S}.csv')
port_mes.to_csv(f'output/resultados/negociacao_portfolio_mes_np{S}.csv', index=False)
(pd.DataFrame({'REC': rec_mensal, 'RCC': rcc_mensal})
   .rename_axis('mes')
   .to_csv(f'output/resultados/negociacao_retorno_mensal_np{S}.csv'))

print(f"Salvo em output/resultados/ (sufixo '{S}', modo {MODO_SAIDA}):")
print(f"  negociacao_pares_np{S}.csv          — resultado por par")
print(f"  eventos_sinal_np{S}.csv             — cada sinal, aprovado ou vetado")
print(f"  negociacao_portfolio_np{S}.csv      — série diária do portfólio")
print(f"  negociacao_portfolio_mes_np{S}.csv  — REC/RCC por portfólio-mês")
print(f"  negociacao_retorno_mensal_np{S}.csv — REC/RCC mensais agregados")